# Model Building

**Models**: Logistic Regression, Decision Tree, Random Forest, SVC and KNN

### 1) Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import precision_recall_curve
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix,classification_report)
from imblearn.over_sampling import SMOTE
from sklearn.inspection import permutation_importance
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

### 2) Load Cleaned Data

In [ ]:
df = pd.read_csv('new_dataset.csv')
print(f" Shape: {df.shape}")
df.head()

### 3) Prepare Features and Target

In [ ]:
all_features = ['age', 'sex', 'bmi', 'systolic_pressure', 'diastolic_pressure',
                'glucose', 'hba1c', 'cholesterol', 'hdl_cholesterol',
                'ldl_cholesterol', 'triglycerides', 'smoking',
                'physical_activity', 'family_history_diabetes', 'family_history_hypertension']

X = df[all_features].copy()
y_diabetes = df['diabetes_risk']
y_hypertension = df['hypertension_risk']

print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"Targets:")
print(f"-Diabetes: {y_diabetes.sum()} positives ({y_diabetes.mean()*100:.1f}%)")
print(f"-Hypertension: {y_hypertension.sum()} positives ({y_hypertension.mean()*100:.1f}%)")

# Single split keeps X and both targets aligned
X_train, X_test, y_diab_train, y_diab_test, y_hyp_train, y_hyp_test = train_test_split(
    X, y_diabetes, y_hypertension, test_size=0.2, random_state=67, stratify=y_diabetes
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

# Feature scaling: apenas as colunas contínuas (as binárias ficam fora).
numeric_features = ['age', 'bmi', 'systolic_pressure', 'diastolic_pressure',
                    'glucose', 'hba1c', 'cholesterol', 'hdl_cholesterol',
                    'ldl_cholesterol', 'triglycerides']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

# SMOTE balancing (apenas para SVM e KNN)
smote_diab = SMOTE(random_state=67)
X_train_balanced_diab, y_diab_train_balanced = smote_diab.fit_resample(X_train_scaled, y_diab_train)

smote_hyp = SMOTE(random_state=67)
X_train_balanced_hyp, y_hyp_train_balanced = smote_hyp.fit_resample(X_train_scaled, y_hyp_train)

### 4) Train Models

In [ ]:
base_models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', random_state=67, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=67, max_depth=10),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=67),
    'SVM': SVC(class_weight='balanced', probability=True, random_state=67),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

models = base_models
diabetes_models = {}
diabetes_thresholds = {}
hypertension_models = {}
hypertension_thresholds = {}
diabetes_results = []
hypertension_results = []

#### Diabetes models


In [ ]:
print("-"*50)
print("Training models for Diabetes:")
print("-"*50)

# Diabetes models
for name, model_template in base_models.items():
    model = clone(model_template)
    
    if name in ['SVM', 'KNN']:
        model.fit(X_train_balanced_diab, y_diab_train_balanced)
    else:
        model.fit(X_train_scaled, y_diab_train)
    
    #get prob
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    #threshold Tuning
    precisions, recalls, thresholds = precision_recall_curve(y_diab_test, y_proba)
    
    f1_scores = np.divide(
        2 * (precisions * recalls),
        (precisions + recalls),
        out=np.zeros_like(precisions),
        where=(precisions + recalls) != 0
    )
    
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    
    #predictions
    y_pred = (y_proba >= best_threshold).astype(int)
    
    #final results
    acc = accuracy_score(y_diab_test, y_pred)
    prec = precision_score(y_diab_test, y_pred)
    rec = recall_score(y_diab_test, y_pred)
    f1 = f1_score(y_diab_test, y_pred)
    auc = roc_auc_score(y_diab_test, y_proba) 
    
    #store results
    diabetes_models[name] = model
    diabetes_thresholds[name] = best_threshold 
    
    diabetes_results.append({
        'Model': name, 
        'Threshold': best_threshold, 
        'Accuracy': acc, 
        'Precision': prec,
        'Recall': rec, 
        'F1-Score': f1, 
        'ROC-AUC': auc
    })
    
    print(f"Best Threshold: {best_threshold:.3f}")
    print(f"Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f} | F1: {f1:.4f}\n")


#### Hypertension models


In [ ]:
print("-"*50)
print("Training models for Hypertension:")
print("-"*50)

# Hypertension models
for name, model_template in base_models.items():
    model = clone(model_template)
    
    if name in ['SVM', 'KNN']:
        model.fit(X_train_balanced_hyp, y_hyp_train_balanced)
    else:
        model.fit(X_train_scaled, y_hyp_train)
    
    #prob
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    #threshold 
    precisions, recalls, thresholds = precision_recall_curve(y_hyp_test, y_proba)
    
    f1_scores = np.divide(
        2 * (precisions * recalls),
        (precisions + recalls),
        out=np.zeros_like(precisions),
        where=(precisions + recalls) != 0
    )
    
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    
    y_pred = (y_proba >= best_threshold).astype(int)
    
    #final res
    acc = accuracy_score(y_hyp_test, y_pred)
    prec = precision_score(y_hyp_test, y_pred)
    rec = recall_score(y_hyp_test, y_pred)
    f1 = f1_score(y_hyp_test, y_pred)
    auc = roc_auc_score(y_hyp_test, y_proba)
    
    hypertension_models[name] = model
    hypertension_thresholds[name] = best_threshold 
    
    hypertension_results.append({
        'Model': name, 
        'Threshold': best_threshold, 
        'Accuracy': acc, 
        'Precision': prec,
        'Recall': rec, 
        'F1-Score': f1, 
        'ROC-AUC': auc
    })
    
    print(f"Best Threshold: {best_threshold:.3f}")
    print(f"Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f} | F1: {f1:.4f}\n")

### 5) Model Comparison

In [ ]:
diab_df = pd.DataFrame(diabetes_results)
hyp_df = pd.DataFrame(hypertension_results)

best_diab_model = diab_df.loc[diab_df['ROC-AUC'].idxmax(), 'Model']
best_diab_auc = diab_df['ROC-AUC'].max()
best_diab_f1 = diab_df.loc[diab_df['ROC-AUC'].idxmax(), 'F1-Score']

best_hyp_model = hyp_df.loc[hyp_df['ROC-AUC'].idxmax(), 'Model']
best_hyp_auc = hyp_df['ROC-AUC'].max()
best_hyp_f1 = hyp_df.loc[hyp_df['ROC-AUC'].idxmax(), 'F1-Score']

print("\n" + "="*60)
print("DIABETES RISK - Models Performance")
print("="*60)
print(diab_df[['Model', 'ROC-AUC', 'F1-Score', 'Recall', 'Precision']].to_string(index=False))

print("\n" + "="*60)
print("HYPERTENSION RISK - Models Performance")
print("="*60)
print(hyp_df[['Model', 'ROC-AUC', 'F1-Score', 'Recall', 'Precision']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Diabetes
ax = axes[0]
diab_sorted = diab_df.sort_values('ROC-AUC', ascending=True)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(diab_sorted)))
bars = ax.barh(diab_sorted['Model'], diab_sorted['ROC-AUC'], color=colors)
ax.set_xlabel('ROC-AUC Score', fontsize=12)
ax.set_title('Diabetes Risk - ROC-AUC por Modelo', fontweight='bold')
ax.set_xlim(0.5, 1.0)
for bar, val in zip(bars, diab_sorted['ROC-AUC']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center')

# Hypertension
ax = axes[1]
hyp_sorted = hyp_df.sort_values('ROC-AUC', ascending=True)
bars = ax.barh(hyp_sorted['Model'], hyp_sorted['ROC-AUC'], color=colors)
ax.set_xlabel('ROC-AUC Score', fontsize=12)
ax.set_title('Hypertension Risk - ROC-AUC por Modelo', fontweight='bold')
ax.set_xlim(0.5, 1.0)
for bar, val in zip(bars, hyp_sorted['ROC-AUC']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center')

plt.tight_layout()
plt.show()

print("BEST MODELS SUMMARY")
print("="*60)
print(f"DIABETES - Best Model: {best_diab_model}")
print(f"           ROC-AUC: {best_diab_auc:.4f}")
print(f"           F1-Score: {best_diab_f1:.4f}")
print(f"\nHYPERTENSION - Best Model: {best_hyp_model}")
print(f"               ROC-AUC: {best_hyp_auc:.4f}")
print(f"               F1-Score: {best_hyp_f1:.4f}")

### Export the data to the .pkl file

In [ ]:
# Export the selected best models and scaler for the Streamlit app
os.makedirs("app/model", exist_ok=True)

bundle = {
    "scaler": scaler,
    "diabetes": diabetes_models[best_diab_model],
    "hypertension": hypertension_models[best_hyp_model],
    "diabetes_name": best_diab_model,
    "hypertension_name": best_hyp_model,
    "diabetes_metrics": {
        "auc": best_diab_auc,
        "f1": best_diab_f1,
        "recall": diab_df.loc[diab_df["Model"] == best_diab_model, "Recall"].values[0],
        "precision": diab_df.loc[diab_df["Model"] == best_diab_model, "Precision"].values[0],
    },
    "hypertension_metrics": {
        "auc": best_hyp_auc,
        "f1": best_hyp_f1,
        "recall": hyp_df.loc[hyp_df["Model"] == best_hyp_model, "Recall"].values[0],
        "precision": hyp_df.loc[hyp_df["Model"] == best_hyp_model, "Precision"].values[0],
    },
}

joblib.dump(bundle, "app/model/models.pkl")
print("Saved model")


### 6) Cross-Validation

In [ ]:
cv_results_diabetes = {}
cv_results_hypertension = {}

scoring_metrics = ['roc_auc', 'accuracy', 'f1', 'precision', 'recall']
metrics_names = ['ROC-AUC', 'Accuracy', 'F1-Score', 'Precision', 'Recall']

print("DIABETES RISK - Cross-validation scores:")
print("-"*50)

for name, model in models.items():
    print(f"\n{name}:")
    cv_scores = {}
    
    for metric in scoring_metrics:
        if name in ['SVM', 'KNN']:
            scores = cross_val_score(model, X_train_balanced_diab, y_diab_train_balanced, 
                                    cv=5, scoring=metric)
        else:
            scores = cross_val_score(model, X_train_scaled, y_diab_train, 
                                    cv=5, scoring=metric)
        
        cv_scores[metric] = scores
        print(f"  {metric.upper()}: {scores.mean():.4f} (±{scores.std():.4f})")
    
    cv_results_diabetes[name] = cv_scores

print("\n" + "-"*50)
print("HYPERTENSION RISK - Cross-validation scores:")
print("-"*50)

for name, model in models.items():
    print(f"\n{name}:")
    cv_scores = {}
    
    for metric in scoring_metrics:
        if name in ['SVM', 'KNN']:
            scores = cross_val_score(model, X_train_balanced_hyp, y_hyp_train_balanced, 
                                    cv=5, scoring=metric)
        else:
            scores = cross_val_score(model, X_train_scaled, y_hyp_train, 
                                    cv=5, scoring=metric)
        
        cv_scores[metric] = scores
        print(f"  {metric.upper()}: {scores.mean():.4f} (±{scores.std():.4f})")
    
    cv_results_hypertension[name] = cv_scores

print("\n" + "="*60)
print("GENERATING BOXPLOTS...")
print("="*60)

# Diabetes boxplots
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('Diabetes Risk - Cross-validation Performance (5-fold)', fontsize=14, fontweight='bold')

for idx, (metric, metric_name) in enumerate(zip(scoring_metrics, metrics_names)):
    ax = axes[idx]
    
    data = []
    labels = []
    
    for name in models.keys():
        scores = cv_results_diabetes[name][metric]
        data.append(scores)
        labels.append(name)
    
    bp = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=False)
    
    for patch in bp['boxes']:
        patch.set_facecolor('white')
        patch.set_edgecolor('black')
        patch.set_alpha(0.7)
    
    for median in bp['medians']:
        median.set_color('red')
        median.set_linewidth(2)
    
    ax.set_ylabel(metric_name, fontsize=11)
    ax.set_title(metric_name, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0.4, 1.05)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.show()

# Hypertension boxplots
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('Hypertension Risk - Cross-validation Performance (5-fold)', fontsize=14, fontweight='bold')

for idx, (metric, metric_name) in enumerate(zip(scoring_metrics, metrics_names)):
    ax = axes[idx]
    
    data = []
    labels = []
    
    for name in models.keys():
        scores = cv_results_hypertension[name][metric]
        data.append(scores)
        labels.append(name)
    
    bp = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=False)
    
    for patch in bp['boxes']:
        patch.set_facecolor('white')
        patch.set_edgecolor('black')
        patch.set_alpha(0.7)
    
    for median in bp['medians']:
        median.set_color('red')
        median.set_linewidth(2)
    
    ax.set_ylabel(metric_name, fontsize=11)
    ax.set_title(metric_name, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0.4, 1.05)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("CROSS-VALIDATION SUMMARY (Mean ± Std)")
print("="*60)

diab_summary = {}
hyp_summary = {}

for name in models.keys():
    diab_summary[name] = {metric: f"{cv_results_diabetes[name][metric].mean():.3f} ± {cv_results_diabetes[name][metric].std():.3f}" 
                          for metric in scoring_metrics}
    hyp_summary[name] = {metric: f"{cv_results_hypertension[name][metric].mean():.3f} ± {cv_results_hypertension[name][metric].std():.3f}" 
                          for metric in scoring_metrics}

diab_df_summary = pd.DataFrame(diab_summary).T
hyp_df_summary = pd.DataFrame(hyp_summary).T

print("\nDIABETES RISK:")
print(diab_df_summary.to_string())

print("\nHYPERTENSION RISK:")
print(hyp_df_summary.to_string())

diab_best_cv = max(cv_results_diabetes.keys(), 
                   key=lambda x: cv_results_diabetes[x]['roc_auc'].mean())
hyp_best_cv = max(cv_results_hypertension.keys(), 
                  key=lambda x: cv_results_hypertension[x]['roc_auc'].mean())

print(f"\nBEST MODEL (CV ROC-AUC):")
print(f"  Diabetes: {diab_best_cv} ({cv_results_diabetes[diab_best_cv]['roc_auc'].mean():.4f})")
print(f"  Hypertension: {hyp_best_cv} ({cv_results_hypertension[hyp_best_cv]['roc_auc'].mean():.4f})")

### ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Diabetes ROC
for name, model in diabetes_models.items():
    if name in ['SVM', 'KNN']:
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_diab_test, y_proba)
    auc = roc_auc_score(y_diab_test, y_proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=1.5)

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Diabetes Risk - ROC Curves', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=8)
axes[0].grid(alpha=0.3)

# Hypertension ROC
for name, model in hypertension_models.items():
    if name in ['SVM', 'KNN']:
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_hyp_test, y_proba)
    auc = roc_auc_score(y_hyp_test, y_proba)
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=1.5)

axes[1].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Hypertension Risk - ROC Curves', fontweight='bold')
axes[1].legend(loc='lower right', fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 7) Feature Importance

In [ ]:
#DIABETES
if best_diab_model == 'Random Forest':
    model = diabetes_models['Random Forest']
    importances = model.feature_importances_
    
elif best_diab_model == 'Logistic Regression':
    model = diabetes_models['Logistic Regression']
    importances = np.abs(model.coef_[0])
    
elif best_diab_model == 'Decision Tree':
    model = diabetes_models['Decision Tree']
    importances = model.feature_importances_
    
else:  # SVM or KNN
    model = diabetes_models[best_diab_model]
    print(f"Calculating permutation importance for {best_diab_model} (Diabetes)...")
    perm_importance = permutation_importance(model, X_test_scaled, y_diab_test,
        n_repeats=10, random_state=67, scoring='roc_auc')
    importances = perm_importance.importances_mean

# Normalize
importances = importances / importances.max()
feature_names = all_features
indices = np.argsort(importances)[::-1]

# Plot Diabetes
plt.figure(figsize=(10, 6))
plt.barh(range(10), importances[indices[:10]])
plt.yticks(range(10), [feature_names[i] for i in indices[:10]])
plt.xlabel('Feature Importance (normalized 0.0 to 1.0)', fontsize=12)
plt.title(f'Diabetes Risk - Top 10 Features ({best_diab_model})', fontweight='bold')
plt.gca().invert_yaxis()
plt.xlim(0, 1.0)
plt.tight_layout()
plt.show()



# HYPERTENSION
if best_hyp_model == 'Random Forest':
    model_hyp = hypertension_models['Random Forest']
    importances_hyp = model_hyp.feature_importances_
    
elif best_hyp_model == 'Logistic Regression':
    model_hyp = hypertension_models['Logistic Regression']
    importances_hyp = np.abs(model_hyp.coef_[0])
    
elif best_hyp_model == 'Decision Tree':
    model_hyp = hypertension_models['Decision Tree']
    importances_hyp = model_hyp.feature_importances_
    
else:  # SVM or KNN 
    model_hyp = hypertension_models[best_hyp_model]
    print(f"Calculating permutation importance for {best_hyp_model} (Hypertension)...")
    perm_importance = permutation_importance(model_hyp, X_test_scaled, y_hyp_test,n_repeats=10, random_state=67, scoring='roc_auc')
    importances_hyp = perm_importance.importances_mean  

# Normalize
importances_hyp = importances_hyp / importances_hyp.max()
indices_hyp = np.argsort(importances_hyp)[::-1]

# Plot Hypertension
plt.figure(figsize=(10, 6))
plt.barh(range(10), importances_hyp[indices_hyp[:10]])
plt.yticks(range(10), [feature_names[i] for i in indices_hyp[:10]])
plt.xlabel('Feature Importance (normalized 0.0 to 1.0)', fontsize=12)
plt.title(f'Hypertension Risk - Top 10 Features ({best_hyp_model})', fontweight='bold')
plt.gca().invert_yaxis()
plt.xlim(0, 1.0)
plt.tight_layout()
plt.show()
